<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Contextual_Retrieval_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contextual Retrieval: Does a Situating Summary Improve Recall?

The production AI Tutor spends one cheap LLM call per chunk at ingest time. The call reads
the whole document, looks at one chunk, and writes a sentence or two describing where that
chunk sits in the document. That sentence is appended to the chunk before embedding. This is
**contextual retrieval**, from Anthropic's September 2024 post, and it is the one ingest stage
Part 1 of this course does not build.

This notebook measures whether it pays. We build three versions of the same index over the same
50-ish-token-to-500-token chunks of public Hugging Face documentation:

| Arm | What gets embedded |
|---|---|
| `raw chunk` | the chunk text, nothing else |
| `+ metadata header` | `Title` / `Source` / `Heading path` header, then the chunk (what `app/chroma_rag.py` stores) |
| `+ situating context` | the header, the chunk, and a `Context:` line written by an LLM (what `add_context_to_nodes.py` produces) |

Then we write a small synthetic question set with a typed LLM call, one question per sampled
chunk, and score all three arms with hit rate and MRR.

**Cost and runtime:** about 650 Gemini Flash-Lite calls and 4 embedding passes, which took two and
a half minutes and cost $0.63 at paid-tier pricing (free on the free tier). Output captured
August 2026; yours may differ. The last cell computes the cost rather than trusting this one.

Companion to the lesson *From Notebook to Production: The Retrieval Service* (Section 13,
Lesson 2).

## Install Packages and Setup Variables

In [1]:
# Shared install profile for this notebook (pin set checked August 2026)
!pip install -q google-genai==2.18.0 openai==3.0.0 tiktoken==0.13.0 \
                pydantic==2.13.4 numpy==2.5.2 pandas==3.0.5

In [2]:
import os

# In Colab, store your key with the key icon in the left sidebar.
# Locally, export GEMINI_API_KEY (or OPENAI_API_KEY) before starting Jupyter.
try:
    from google.colab import userdata

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    # os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

assert os.environ.get("GEMINI_API_KEY"), "Set GEMINI_API_KEY before running this notebook."

## The Provider Setup Cell

The same two helpers every retrieval notebook in this course uses: `generate_typed()` for a
schema-constrained chat call, and `embed_texts()` for a batch of embeddings. Set `PROVIDER`
and the rest of the notebook stays provider-neutral.

Two notes on providers:

- **Anthropic has no embedding API.** Anthropic's own docs say so and point to third parties.
  A student who prefers Claude for chat still picks Gemini or OpenAI for embeddings, so this
  notebook offers those two.
- **`gemini-embedding-001` does not renormalize truncated vectors.** We ask for 768 dimensions
  instead of the default 3072, which is Matryoshka truncation, so we divide by the L2 norm
  ourselves. The newer `gemini-embedding-2` renormalizes for you.

The OpenAI branch below is provided for reference. The saved outputs all come from the Gemini
path, so check the OpenAI path yourself before relying on it.

In [3]:
PROVIDER = "gemini"  # "gemini" or "openai"

import time

import numpy as np
from pydantic import BaseModel, Field

if PROVIDER == "gemini":
    from google import genai
    from google.genai import types

    CHAT_MODEL = "gemini-3.5-flash-lite"
    EMBED_MODEL = "gemini-embedding-001"
    EMBED_DIM = 768
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    def _generate_typed(prompt: str, schema: type[BaseModel], max_output_tokens: int):
        response = client.models.generate_content(
            model=CHAT_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=max_output_tokens,
                response_mime_type="application/json",
                response_schema=schema,
            ),
        )
        usage = response.usage_metadata
        parsed = response.parsed
        if not isinstance(parsed, schema):
            parsed = schema.model_validate_json(response.text)
        return parsed, (usage.prompt_token_count, usage.candidates_token_count or 0)

    def _embed_batch(texts: list[str], is_query: bool) -> list[list[float]]:
        response = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts,
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_QUERY" if is_query else "RETRIEVAL_DOCUMENT",
                output_dimensionality=EMBED_DIM,
            ),
        )
        return [embedding.values for embedding in response.embeddings]

elif PROVIDER == "openai":
    # Provided for reference; check before relying on it.
    from openai import OpenAI

    CHAT_MODEL = "gpt-5.6-luna"
    EMBED_MODEL = "text-embedding-3-small"
    EMBED_DIM = 768
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    def _generate_typed(prompt: str, schema: type[BaseModel], max_output_tokens: int):
        response = client.responses.parse(
            model=CHAT_MODEL,
            input=prompt,
            text_format=schema,
            max_output_tokens=max_output_tokens,
        )
        usage = response.usage
        return response.output_parsed, (usage.input_tokens, usage.output_tokens)

    def _embed_batch(texts: list[str], is_query: bool) -> list[list[float]]:
        response = client.embeddings.create(
            model=EMBED_MODEL, input=texts, dimensions=EMBED_DIM
        )
        return [item.embedding for item in response.data]

else:
    raise ValueError(f"Unknown PROVIDER {PROVIDER!r}")

TOKENS_USED = {"input": 0, "output": 0}


def generate_typed(prompt: str, schema: type[BaseModel], max_output_tokens: int = 1024):
    """One schema-constrained call, with retry on transient API errors."""
    for attempt in range(6):
        try:
            parsed, (prompt_tokens, output_tokens) = _generate_typed(
                prompt, schema, max_output_tokens
            )
            TOKENS_USED["input"] += prompt_tokens
            TOKENS_USED["output"] += output_tokens
            return parsed
        except Exception as exc:
            if attempt == 5:
                raise
            print(f"  retry {attempt}: {type(exc).__name__} {str(exc)[:70]}")
            time.sleep(2 + 2**attempt)


def embed_texts(texts: list[str], is_query: bool = False, batch_size: int = 20) -> np.ndarray:
    """Embed a list of texts in batches, with retry, returning unit-norm rows."""
    vectors: list[list[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        for attempt in range(6):
            try:
                vectors.extend(_embed_batch(batch, is_query))
                break
            except Exception as exc:
                if attempt == 5:
                    raise
                print(f"  embed retry {attempt}: {type(exc).__name__} {str(exc)[:70]}")
                time.sleep(2 + 2**attempt)
    matrix = np.array(vectors, dtype=np.float32)
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


print(f"provider={PROVIDER} chat={CHAT_MODEL} embed={EMBED_MODEL} dim={EMBED_DIM}")

provider=gemini chat=gemini-3.5-flash-lite embed=gemini-embedding-001 dim=768


## The Corpus

Twenty-five pages of public library documentation, pinned to exact commit SHAs so the experiment
is reproducible. They come from four Hugging Face libraries that talk about the same things:
PEFT, TRL, Transformers, and Accelerate. The overlap is the point. Four separate pages explain
FSDP, three explain DeepSpeed, three explain quantization, and six are preference trainers that
share most of their vocabulary. Neighbouring documents are what make retrieval hard, and hard
retrieval is the only condition under which contextual retrieval has anything to prove.

These are the same public documentation sources the production tutor indexes.

In [4]:
import requests

PEFT = "https://raw.githubusercontent.com/huggingface/peft/9f1fe21d8131a24634d6d23c13efa2aae72b6cca/docs/source"
TRL = "https://raw.githubusercontent.com/huggingface/trl/922dc584664d87482935e1fa7d958930fc5223cf/docs/source"
TFM = "https://raw.githubusercontent.com/huggingface/transformers/71c6f699ac9b3f8fc42a6a3e9dc59034c349a678/docs/source/en"
ACC = "https://raw.githubusercontent.com/huggingface/accelerate/7fccde20d2a7a1242633ce6912dcab584738c918/docs/source"

DOCS = [
    {"doc_id": "peft/quicktour", "title": "PEFT quicktour", "source": "peft", "url": f"{PEFT}/quicktour.md"},
    {"doc_id": "peft/quantization", "title": "PEFT quantization", "source": "peft", "url": f"{PEFT}/developer_guides/quantization.md"},
    {"doc_id": "peft/troubleshooting", "title": "PEFT troubleshooting", "source": "peft", "url": f"{PEFT}/developer_guides/troubleshooting.md"},
    {"doc_id": "peft/fsdp", "title": "PEFT with FSDP", "source": "peft", "url": f"{PEFT}/accelerate/fsdp.md"},
    {"doc_id": "peft/deepspeed", "title": "PEFT with DeepSpeed", "source": "peft", "url": f"{PEFT}/accelerate/deepspeed.md"},
    {"doc_id": "trl/dpo_trainer", "title": "TRL DPOTrainer", "source": "trl", "url": f"{TRL}/dpo_trainer.md"},
    {"doc_id": "trl/kto_trainer", "title": "TRL KTOTrainer", "source": "trl", "url": f"{TRL}/kto_trainer.md"},
    {"doc_id": "trl/cpo_trainer", "title": "TRL CPOTrainer", "source": "trl", "url": f"{TRL}/cpo_trainer.md"},
    {"doc_id": "trl/orpo_trainer", "title": "TRL ORPOTrainer", "source": "trl", "url": f"{TRL}/orpo_trainer.md"},
    {"doc_id": "trl/online_dpo_trainer", "title": "TRL OnlineDPOTrainer", "source": "trl", "url": f"{TRL}/online_dpo_trainer.md"},
    {"doc_id": "trl/reward_trainer", "title": "TRL RewardTrainer", "source": "trl", "url": f"{TRL}/reward_trainer.md"},
    {"doc_id": "trl/sft_trainer", "title": "TRL SFTTrainer", "source": "trl", "url": f"{TRL}/sft_trainer.md"},
    {"doc_id": "trl/reducing_memory", "title": "TRL reducing memory usage", "source": "trl", "url": f"{TRL}/reducing_memory_usage.md"},
    {"doc_id": "transformers/cache_explanation", "title": "Transformers KV cache explained", "source": "transformers", "url": f"{TFM}/cache_explanation.md"},
    {"doc_id": "transformers/kv_cache", "title": "Transformers cache classes", "source": "transformers", "url": f"{TFM}/kv_cache.md"},
    {"doc_id": "transformers/attention_interface", "title": "Transformers attention interface", "source": "transformers", "url": f"{TFM}/attention_interface.md"},
    {"doc_id": "transformers/peft", "title": "Transformers PEFT adapters", "source": "transformers", "url": f"{TFM}/peft.md"},
    {"doc_id": "transformers/bitsandbytes", "title": "Transformers bitsandbytes quantization", "source": "transformers", "url": f"{TFM}/quantization/bitsandbytes.md"},
    {"doc_id": "transformers/quantization_concepts", "title": "Transformers quantization concepts", "source": "transformers", "url": f"{TFM}/quantization/concept_guide.md"},
    {"doc_id": "transformers/fsdp", "title": "Transformers FSDP", "source": "transformers", "url": f"{TFM}/fsdp.md"},
    {"doc_id": "transformers/deepspeed", "title": "Transformers DeepSpeed", "source": "transformers", "url": f"{TFM}/deepspeed.md"},
    {"doc_id": "accelerate/fsdp", "title": "Accelerate FSDP", "source": "accelerate", "url": f"{ACC}/usage_guides/fsdp.md"},
    {"doc_id": "accelerate/deepspeed", "title": "Accelerate DeepSpeed", "source": "accelerate", "url": f"{ACC}/usage_guides/deepspeed.md"},
    {"doc_id": "accelerate/fsdp_vs_deepspeed", "title": "Accelerate FSDP vs DeepSpeed", "source": "accelerate", "url": f"{ACC}/concept_guides/fsdp_and_deepspeed.md"},
    {"doc_id": "accelerate/gradient_accumulation", "title": "Accelerate gradient accumulation", "source": "accelerate", "url": f"{ACC}/usage_guides/gradient_accumulation.md"},
]

for doc in DOCS:
    doc["content"] = requests.get(doc["url"], timeout=30).text
print(f"{len(DOCS)} documents, {sum(len(d['content']) for d in DOCS):,} chars")
for source in dict.fromkeys(d["source"] for d in DOCS):
    print(f"  {source:14s} {sum(1 for d in DOCS if d['source'] == source):>2} pages")

25 documents, 387,088 chars
  peft            5 pages
  trl             8 pages
  transformers    8 pages
  accelerate      4 pages


## Heading-Aware Chunking

A simplified version of `heading_aware_markdown_chunks` from `app/chroma_rag.py`. The rules that
matter: a fenced code block is one unit and is never split, a heading change forces a chunk
boundary, and units accumulate until they would exceed the token budget.

Production uses 800-token chunks over a 3,000-document corpus. We use 500 here so that
twenty-five documents produce enough chunks for ranking to be interesting.

In [5]:
import re

import tiktoken

ENCODING = tiktoken.get_encoding("cl100k_base")
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*#*\s*$")
FENCE_RE = re.compile(r"^\s*(`{3,}|~{3,})")


def n_tokens(text: str) -> int:
    return len(ENCODING.encode(text, disallowed_special=()))


def parse_units(text: str, default_path: tuple[str, ...] = ()):
    """Split markdown into (text, heading_path) units: headings, code fences, paragraphs."""
    lines, units, stack, buffer = text.splitlines(), [], [], []

    def path():
        return tuple(stack) or default_path

    def flush_paragraphs():
        nonlocal buffer
        paragraph = []
        for line in buffer:
            if line.strip():
                paragraph.append(line)
            elif paragraph:
                units.append(("\n".join(paragraph).strip(), path()))
                paragraph = []
        if paragraph:
            units.append(("\n".join(paragraph).strip(), path()))
        buffer = []

    index = 0
    while index < len(lines):
        line = lines[index]
        fence = FENCE_RE.match(line)
        if fence:  # a code block is one indivisible unit
            flush_paragraphs()
            marker, block = fence.group(1), [line]
            index += 1
            while index < len(lines):
                block.append(lines[index])
                closing = FENCE_RE.match(lines[index])
                if closing and closing.group(1)[0] == marker[0]:
                    index += 1
                    break
                index += 1
            units.append(("\n".join(block).strip(), path()))
            continue
        heading = HEADING_RE.match(line)
        if heading:  # a heading resets the path at its level
            flush_paragraphs()
            stack = stack[: len(heading.group(1)) - 1] + [heading.group(2).strip()]
            units.append((line.strip(), path()))
            index += 1
            continue
        buffer.append(line)
        index += 1
    flush_paragraphs()
    return [unit for unit in units if unit[0].strip()]


def heading_aware_chunks(text: str, title: str, chunk_size: int = 500):
    chunks, parts, current_path, size = [], [], (), 0

    def flush():
        nonlocal parts, current_path, size
        body = "\n\n".join(part for part in parts if part.strip()).strip()
        if body:
            chunks.append({"text": body, "heading_path": " > ".join(current_path)})
        parts, current_path, size = [], (), 0

    for unit_text, unit_path in parse_units(text, (title,)):
        unit_tokens = n_tokens(unit_text)
        if (current_path and unit_path != current_path) or (parts and size + unit_tokens > chunk_size):
            flush()
        if not parts:
            current_path = unit_path
        parts.append(unit_text)
        size += unit_tokens
    flush()
    return chunks

Now build the chunk records. `format_chunk_for_retrieval` does the same job as production's
function: it prepends a small metadata header so the embedded text carries the document title,
the source key, and the heading path. We drop chunks under 40 tokens (bare headings) and the
license banner every Hugging Face doc starts with.

In [6]:
def format_chunk_for_retrieval(text: str, metadata: dict) -> str:
    """Metadata header + chunk text. Mirrors app/chroma_rag.py."""
    header = [f"Title: {metadata['title']}", f"Source: {metadata['source']}"]
    if metadata.get("heading_path"):
        header.append(f"Heading path: {metadata['heading_path']}")
    return "\n".join(header) + "\n\n" + text.strip()


records = []
for doc in DOCS:
    for index, chunk in enumerate(heading_aware_chunks(doc["content"], doc["title"])):
        if n_tokens(chunk["text"]) < 40 or "Apache License" in chunk["text"]:
            continue
        records.append(
            {
                "chunk_id": f"{doc['doc_id']}:{index}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "source": doc["source"],
                "heading_path": chunk["heading_path"],
                "text": chunk["text"],
                "tokens": n_tokens(chunk["text"]),
            }
        )

from collections import Counter

print(f"{len(records)} chunks")
print(f"median tokens: {int(np.median([r['tokens'] for r in records]))}")
for doc_id, count in Counter(r["doc_id"] for r in records).most_common():
    print(f"  {doc_id:38s} {count:>3}")

348 chunks
median tokens: 232
  peft/deepspeed                          22
  trl/sft_trainer                         22
  accelerate/deepspeed                    22
  peft/troubleshooting                    21
  trl/kto_trainer                         21
  trl/dpo_trainer                         19
  peft/quantization                       18
  transformers/bitsandbytes               16
  peft/fsdp                               15
  trl/cpo_trainer                         14
  transformers/kv_cache                   14
  transformers/attention_interface        14
  trl/reducing_memory                     13
  accelerate/fsdp_vs_deepspeed            13
  trl/online_dpo_trainer                  12
  trl/reward_trainer                      12
  transformers/peft                       12
  accelerate/gradient_accumulation        12
  transformers/quantization_concepts      10
  transformers/deepspeed                  10
  trl/orpo_trainer                         9
  peft/quicktour         

## Contextual Retrieval at Ingest

Here is the stage Part 1 skipped. For every chunk we send the model the whole document plus that
one chunk, and ask for a short situating summary. The prompt and the Pydantic schema below are the
ones the production pipeline uses in `data/scraping_scripts/add_context_to_nodes.py`.

Two production details are worth copying. The schema is a typed structured output, the same
pattern Section 6 taught, so we never parse prose. And every call goes through a retry wrapper,
because a few thousand chunks against a rate-limited API will hit a 429 sooner or later.

We run the calls through a small thread pool. Production uses `asyncio` with a semaphore and a
token-per-minute limiter. Eight threads is the notebook-sized version of the same idea.

In [7]:
from concurrent.futures import ThreadPoolExecutor


class SituatedContext(BaseModel):
    title: str = Field(..., description="The title of the document.")
    context: str = Field(
        ..., description="The context to situate the chunk within the document."
    )


SITUATE_PROMPT = """<document>
{doc}
</document>

Here is the chunk we want to situate within the whole document above:

<chunk>
{chunk}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Return a title for the document and the succinct context."""


def situate_chunk(record: dict, documents: dict[str, str]) -> str:
    prompt = SITUATE_PROMPT.format(doc=documents[record["doc_id"]], chunk=record["text"])
    return generate_typed(prompt, SituatedContext).context


documents = {doc["doc_id"]: doc["content"] for doc in DOCS}
started = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    contexts = list(pool.map(lambda r: situate_chunk(r, documents), records))

for record, context in zip(records, contexts):
    record["header_text"] = format_chunk_for_retrieval(record["text"], record)
    record["contextual_text"] = f"{record['header_text']}\n\nContext: {context}"

print(f"situated {len(records)} chunks in {time.time() - started:.0f}s")
print(f"tokens so far: {TOKENS_USED}")

situated 348 chunks in 46s
tokens so far: {'input': 1747195, 'output': 19655}


Look at one chunk before and after. This is the whole idea in one screen: the chunk itself never
names the trainer or the argument it belongs to, and the appended `Context:` line does.

In [8]:
# The KTO trainer's Mixture-of-Experts section: a chunk whose text never names its trainer.
sample = next(r for r in records if r["chunk_id"] == "trl/kto_trainer:19")
print("--- raw chunk " + "-" * 60)
print(sample["text"][:420])
print("\n--- what actually gets embedded " + "-" * 40)
print(sample["contextual_text"][-600:])

--- raw chunk ------------------------------------------------------------
### For Mixture of Experts Models: Enabling the auxiliary loss

MOEs are the most efficient if the load is about equally distributed between experts.  
To ensure that we train MOEs similarly during preference-tuning, it is beneficial to add the auxiliary loss from the load balancer to the final loss.

This option is enabled by setting `output_router_logits=True` in the model config (e.g. [`~transformers.MixtralConfig

--- what actually gets embedded ----------------------------------------
g, it is beneficial to add the auxiliary loss from the load balancer to the final loss.

This option is enabled by setting `output_router_logits=True` in the model config (e.g. [`~transformers.MixtralConfig`]).  
To scale how much the auxiliary loss contributes to the total loss, use the hyperparameter `router_aux_loss_coef=...` (default: `0.001`) in the model config.

Context: The document is the user guide for the KTO Traine

## Writing the Question Set

An evaluation needs questions with known right answers. We take every chunk long enough to contain
an answerable fact and ask the model to write one question for it, again as a typed call. The
rules in the prompt matter more than the model: short, no library or page names, no copied
identifiers. A question that quotes the chunk's own vocabulary measures string overlap, not
retrieval.

The gold label is the chunk the question was written from. We use every eligible chunk rather than
a sample because the differences we are hunting are small, and a small question set cannot see
them.

<b>Caveat to keep in view:</b> questions generated from a chunk are easier than questions real
students ask, because they are guaranteed answerable and guaranteed to have exactly one right
chunk. The production harness uses 60 real academy questions for this reason. A synthetic set is
the cheap version, good for ranking two indexes against each other and bad for absolute claims.

In [9]:
class StudentQuestion(BaseModel):
    question: str = Field(..., description="One short question a student would ask.")


QUESTION_PROMPT = """You are writing evaluation questions for a retrieval system over AI library documentation.

<passage>
{chunk}
</passage>

Write ONE short question that this passage answers.
Rules:
- At most 12 words. Write it the way a developer types into a chat box, mid-task.
- Do NOT name the library, the page, or the section heading.
- Do NOT copy identifiers, class names, or distinctive phrases from the passage.
- Ask about the idea in ordinary words, not the passage's wording."""

# Long enough to contain an answerable fact.
sampled = [record for record in records if record["tokens"] >= 90]

with ThreadPoolExecutor(max_workers=8) as pool:
    written = list(
        pool.map(
            lambda r: generate_typed(
                QUESTION_PROMPT.format(chunk=r["text"]), StudentQuestion, 512
            ).question,
            sampled,
        )
    )

questions = [
    {"question": q, "gold_chunk_id": r["chunk_id"]} for q, r in zip(written, sampled)
]
print(f"{len(questions)} questions\n")
for item in questions[:8]:
    print(f"  {item['question']:<62s} -> {item['gold_chunk_id']}")

  retry 0: ServerError 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is cu


304 questions

  How can I update large models with fewer trainable weights?    -> peft/quicktour:1
  What parameters go into setting up a weight adaptation config? -> peft/quicktour:2
  How do I check the number of trainable parameters?             -> peft/quicktour:3
  How do I start training after setting up the adapters?         -> peft/quicktour:4
  How do I store my trained model files?                         -> peft/quicktour:5
  How do I load a trained adapter onto a base model?             -> peft/quicktour:6
  How do I switch between multiple loaded models?                -> peft/quicktour:7
  How do I load a trained model for inference?                   -> peft/quicktour:8


## Embedding the Three Arms

We embed the same chunks with the same model, and only the text of each arm differs. Documents
get the `RETRIEVAL_DOCUMENT` task type and questions get `RETRIEVAL_QUERY`, which is the Gemini
equivalent of Cohere's `input_type="search_document"` / `"search_query"` split that
`app/chroma_rag.py` uses. Asymmetric embedding models are trained to place a question near the
passage that answers it rather than near other questions, so getting this parameter wrong quietly
costs recall.

In [10]:
arms = {
    "raw chunk": [r["text"] for r in records],
    "+ metadata header": [r["header_text"] for r in records],
    "+ situating context": [r["contextual_text"] for r in records],
}

arm_vectors = {}
for name, texts in arms.items():
    arm_vectors[name] = embed_texts(texts, is_query=False)
    print(f"{name:22s} {arm_vectors[name].shape}")

question_vectors = embed_texts([q["question"] for q in questions], is_query=True)
print(f"{'questions':22s} {question_vectors.shape}")

raw chunk              (348, 768)


+ metadata header      (348, 768)


+ situating context    (348, 768)


questions              (304, 768)


## Scoring: Hit Rate and MRR

We use two metrics, both computed over the ranked list a real retriever would hand the reranker.

- **Hit rate@k** answers "was the right chunk in the top k at all?" It is the same question
  `recall@shown` asks in `evals/grade.py`.
- **MRR** (mean reciprocal rank) is a ranking score: the gold chunk at position 1 scores 1.0, at
  position 2 scores 0.5, at position 3 scores 0.33, and past position 10 scores 0.

Hit rate@5 is the one that maps to production, because the retrieval service reranks to the top 5.

In [11]:
import pandas as pd

chunk_ids = [r["chunk_id"] for r in records]
gold_rows = [chunk_ids.index(q["gold_chunk_id"]) for q in questions]


def gold_positions(document_vectors: np.ndarray) -> list[int]:
    """1-based rank of each question's gold chunk under cosine similarity."""
    ranking = np.argsort(-(question_vectors @ document_vectors.T), axis=1)
    return [int(np.where(row == gold)[0][0]) + 1 for row, gold in zip(ranking, gold_rows)]


def score(positions: list[int]) -> dict:
    n = len(positions)
    return {
        "hit@1": sum(p <= 1 for p in positions) / n,
        "hit@3": sum(p <= 3 for p in positions) / n,
        "hit@5": sum(p <= 5 for p in positions) / n,
        "MRR@10": sum(1 / p if p <= 10 else 0.0 for p in positions) / n,
    }


positions = {name: gold_positions(vectors) for name, vectors in arm_vectors.items()}
report = pd.DataFrame({name: score(pos) for name, pos in positions.items()}).T
print(f"{len(questions)} questions over {len(records)} chunks\n")
report.round(3)

304 questions over 348 chunks



,hit@1,hit@3,hit@5,MRR@10
raw chunk,0.474,0.678,0.786,0.597
+ metadata header,0.470,0.684,0.773,0.597
+ situating context,0.470,0.694,0.780,0.600


Now the part a summary table hides: which individual questions moved. A metric that improves by
five points on 40 questions moved two questions, and you should always look at which two.

In [12]:
header, contextual = positions["+ metadata header"], positions["+ situating context"]
moved = pd.DataFrame(
    [
        {"question": q["question"], "header_rank": h, "contextual_rank": c, "delta": c - h}
        for q, h, c in zip(questions, header, contextual)
        if h != c
    ]
).sort_values("delta")

better = sum(c < h for h, c in zip(header, contextual))
worse = sum(c > h for h, c in zip(header, contextual))
print(f"improved {better}, worsened {worse}, unchanged {len(questions) - better - worse}")
moved

improved 70, worsened 53, unchanged 181


,question,header_rank,contextual_rank,delta
47,What are the main drawbacks of standard superv...,70,24,-46
109,What is the memory offload device for parameters?,109,75,-34
42,Can I use PEFT models with the optimized kerne...,92,72,-20
39,How can I speed up model training while using ...,38,21,-17
86,How do I check the memory usage of a loaded mo...,24,14,-10
...,...,...,...,...
111,How do I configure distributed training with m...,38,51,13
116,How do I pass custom settings for distributed ...,13,29,16
97,Where can I read about reducing memory usage f...,15,34,19
95,How do I offload the optimizer to CPU memory?,28,49,21


## Where the Remaining Errors Are

Before reading anything into the gaps between arms, find out what the failures look like. Score the
same rankings at document level: did the top result at least come from the right page, even when it
was the wrong chunk of that page?

In [13]:
doc_ids = [r["doc_id"] for r in records]
gold_docs = [records[g]["doc_id"] for g in gold_rows]


def document_hit_rate(document_vectors: np.ndarray, k: int) -> float:
    ranking = np.argsort(-(question_vectors @ document_vectors.T), axis=1)[:, :k]
    return float(
        np.mean([gold in [doc_ids[i] for i in row] for row, gold in zip(ranking, gold_docs)])
    )


diagnostic = pd.DataFrame(
    {
        name: {
            "chunk hit@1": score(positions[name])["hit@1"],
            "document hit@1": document_hit_rate(vectors, 1),
            "document hit@5": document_hit_rate(vectors, 5),
        }
        for name, vectors in arm_vectors.items()
    }
).T
diagnostic.round(3)

,chunk hit@1,document hit@1,document hit@5
raw chunk,0.474,0.625,0.878
+ metadata header,0.470,0.609,0.849
+ situating context,0.470,0.625,0.859


## Is the Difference Real?

A table of four decimal numbers invites you to read a 0.02 gap as a result. Before you do, ask what
the same experiment would produce if you ran it again. Two sources of noise are in play: the
question set is small, and the model that wrote the questions and the situating summaries is not
deterministic even at `temperature=0`.

The first source we can measure right now. Bootstrap resampling over the questions gives a 95%
interval for the paired difference in MRR: draw the question set with replacement 2,000 times and
look at the spread of the difference. If that interval straddles zero, this experiment cannot tell
the arms apart, whatever the point estimate says.

In [14]:
def paired_bootstrap(arm_a: str, arm_b: str, draws: int = 2000, seed: int = 0):
    """95% interval for MRR(arm_b) - MRR(arm_a), resampling questions."""
    rng = np.random.default_rng(seed)
    reciprocal_a = np.array([1 / p if p <= 10 else 0.0 for p in positions[arm_a]])
    reciprocal_b = np.array([1 / p if p <= 10 else 0.0 for p in positions[arm_b]])
    n = len(reciprocal_a)
    deltas = [
        (reciprocal_b[idx] - reciprocal_a[idx]).mean()
        for idx in (rng.integers(0, n, n) for _ in range(draws))
    ]
    low, high = np.percentile(deltas, [2.5, 97.5])
    return reciprocal_b.mean() - reciprocal_a.mean(), low, high


for arm_a, arm_b in [
    ("raw chunk", "+ metadata header"),
    ("raw chunk", "+ situating context"),
    ("+ metadata header", "+ situating context"),
]:
    point, low, high = paired_bootstrap(arm_a, arm_b)
    verdict = "excludes 0" if low > 0 or high < 0 else "straddles 0"
    print(f"{arm_b:>21s} - {arm_a:<21s} MRR {point:+.3f}  95% CI [{low:+.3f}, {high:+.3f}]  {verdict}")

    + metadata header - raw chunk             MRR -0.001  95% CI [-0.028, +0.025]  straddles 0
  + situating context - raw chunk             MRR +0.003  95% CI [-0.027, +0.030]  straddles 0
  + situating context - + metadata header     MRR +0.004  95% CI [-0.017, +0.023]  straddles 0


## What the Ingest Call Cost

The situating pass sends the entire document with every chunk, so its input token count is roughly
the corpus size times the average chunks per document. That is the honest price of contextual
retrieval, and it is why production runs it on a Flash-Lite-tier model and caches the result in a
pickle instead of recomputing it per query.

In [15]:
# Gemini 3.5 Flash-Lite paid-tier list price, checked August 2026. Per 1M tokens.
PRICE_INPUT, PRICE_OUTPUT = 0.30, 2.50

input_tokens, output_tokens = TOKENS_USED["input"], TOKENS_USED["output"]
corpus_tokens = sum(n_tokens(doc["content"]) for doc in DOCS)
cost = (input_tokens * PRICE_INPUT + output_tokens * PRICE_OUTPUT) / 1_000_000

print(f"corpus:                {corpus_tokens:>10,} tokens")
print(f"ingest input tokens:   {input_tokens:>10,}  ({input_tokens / corpus_tokens:.0f}x the corpus)")
print(f"ingest output tokens:  {output_tokens:>10,}")
print(f"per chunk:             {input_tokens / len(records):>10,.0f} input tokens")
print(f"estimated cost:        {cost:>10.3f} USD  ({cost / len(records) * 1000:.2f} USD per 1k chunks)")

corpus:                    93,505 tokens
ingest input tokens:    1,878,764  (20x the corpus)
ingest output tokens:      25,622
per chunk:                  5,399 input tokens
estimated cost:             0.628 USD  (1.80 USD per 1k chunks)


## Reading the Result Honestly

The honest summary of this run: the three arms sit inside each other's noise. The situating context
comes out nominally best on MRR (0.600 against 0.597 for raw chunks) and on hit@3, while the raw
chunk wins hit@1 and hit@5. The bootstrap interval for that MRR gap runs from -0.027 to +0.030. An
interval containing zero means this experiment cannot separate the arms. Report the interval, not
the point estimate.

Five things to take away.

**Tripling the corpus tightened the interval and did not open the gap.** The same three arms over
114 chunks of eight pages put the situating advantage at +0.023 MRR with an interval of
[-0.033, +0.079]. At 348 chunks over twenty-five pages it is +0.003 with an interval of
[-0.027, +0.030]. The interval is half as wide and the point estimate moved toward zero, which is
what a genuine null looks like as precision improves.

**The bigger corpus is measurably harder, which was the point of growing it.** Four of these pages
explain FSDP, three explain DeepSpeed, three explain quantization, and six are preference trainers.
Document hit@5 drops to 0.85-0.88 here from 0.95-0.98 on the eight-page corpus, so the retriever
really is confusing documents more often. The situating summaries had more to disambiguate and
still bought nothing measurable.

**A null result is a result, and this one has a mechanism.** The document-level diagnostic explains
it. The right page reaches the top 5 about 85 to 88 percent of the time in every arm, while the
right chunk is first about 47 percent of the time. The errors that remain are mostly intra-document:
the correct page, the wrong section of it. A situating summary mostly restates which document and
section a chunk belongs to, which is exactly the part the ranking already gets right.

**Rerun this and the numbers move.** Point estimates shift by a few points of hit@1 between runs
and the arms can change places, because the model writing the questions and the summaries is not
deterministic even at `temperature=0`. The qualitative conclusion holds: everything inside the
noise. That stability is the part to trust.

**The cost is real and it only points one way.** The ingest pass billed 20 times the corpus in
input tokens, because every chunk ships its entire document to the model. That is about $1.80 per
thousand chunks at Flash-Lite prices. You pay it once and reuse it forever, which is why production
runs it, and you should still be able to point at a measurement that justifies it on your corpus.

Where does the technique pay off, then? Anthropic reports a 35 percent reduction in top-20
retrieval failures from contextual embeddings, 49 percent once contextual BM25 is added, and 67
percent with reranking, measured on corpora far larger than twenty-five pages, where picking the
right document out of thousands is the hard part and a chunk's provenance carries real information.
Twenty-five pages are not that setting. The production tutor's 3,000 documents and 8 million tokens
are.

## Things to Try

- **Add BM25 and fuse.** Anthropic's larger win comes from contextual embeddings plus contextual
  BM25 over Reciprocal Rank Fusion, and only the dense half is measured above. You built RRF in
  Part 1. Drop it in and add a fourth arm. This is the most likely place for the gap to appear.
- **Grow the corpus again, in a new direction.** Twenty-five pages of one library family did not
  move the result. Add a hundred pages from an unrelated domain and watch whether document-level
  confusion, rather than section-level confusion, becomes the dominant error.
- **Move the context to the front.** Production appends `Context:` after the chunk. Anthropic
  prepends it. Nobody has measured which is better on this corpus.
- **Swap the situating model.** Rerun the situating pass with a Flash-tier or Pro-tier model and
  see whether a better summary buys any recall at ten times the ingest price.
- **Break the header.** Set every document title to `"Untitled"` and rerun. The gap between the
  header arm and the contextual arm should widen, which tells you what the LLM call is actually
  compensating for.
- **Use real questions.** Replace the generated set with twenty questions you write yourself
  before looking at the chunks, and see how much the absolute numbers drop.